## 임데빙 적재

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day02"

if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

from app.core.config import get_settings

url = get_settings().database_url
print("프로젝트 루트 :", ROOT)
print("붙은 DB       :", url.split("@")[-1] if "@" in url else url)

프로젝트 루트 : c:\workspace\hanwha-agent
붙은 DB       : localhost:5432/agent


In [2]:
from app.models.document import Chunk
from app.db.session import session_scope
from app.models.document import Chunk
from sqlalchemy import func, select

queries = ['숙박', '숙박비', '출장 갈 때 숙박비 얼마?', '출장 숙박비']
zeros = []
with session_scope() as s:
    for q in queries:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{q}%'))) or 0
        print(f'{q!r:<28} → {n}줄')
        if n == 0:
            zeros.append(q)
print()
print(f'넷 중 {len(zeros)}개가 0 줄입니다.')


'숙박'                         → 0줄
'숙박비'                        → 0줄
'출장 갈 때 숙박비 얼마?'             → 0줄
'출장 숙박비'                     → 0줄

넷 중 4개가 0 줄입니다.


In [3]:
# 노트북 
from sqlalchemy import text
from app.db.session import session_scope
with session_scope() as s:
    v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
    if v:
        print(f'vector 확장 : 이미 켜져 있습니다 — {v}')
    else:
        s.execute(text('CREATE EXTENSION IF NOT EXISTS vector'))
        s.commit()
        v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
        print(f'vector 확장 : 켰습니다 — {v}')

vector 확장 : 이미 켜져 있습니다 — 0.8.6


In [6]:
import importlib
from pathlib import Path
import app.models.base
importlib.reload(app.models.base)
from app.core.config import get_settings
get_settings.cache_clear()
settings = get_settings()
print('models/base.py 의 EMBED_DIM :', app.models.base.EMBED_DIM)
print('settings.embed_dim          :', settings.embed_dim)
print('settings.embed_provider     :', settings.embed_provider)
print('settings.embed_model_dir    :', settings.embed_model_dir)
print('settings.upstage_embed_model:', settings.upstage_embed_model)
print('.env :', f'{len(Path('.env').read_text(encoding='utf-8').splitlines())}줄')

models/base.py 의 EMBED_DIM : 1024
settings.embed_dim          : 1024
settings.embed_provider     : local
settings.embed_model_dir    : models/bge-m3
settings.upstage_embed_model: 
.env : 74줄
